# Прогон атак на живой модели — Google Colab

Блокнот сам ставит Ollama, скачивает модель и бенчмарки, прогоняет атаки **без фильтра и с фильтром**
и сохраняет результаты на ваш **Google Диск** (папка `vkr_live_runs`).

**Перед запуском:**
1. Меню **Среда выполнения → Сменить среду выполнения → T4 GPU → Сохранить**.
2. В ячейке 1 выберите модель и вариант (сначала всегда `test`).
3. Меню **Среда выполнения → Выполнить все** (Ctrl+F9). Разрешите доступ к Google Диску, когда спросят.

**Если сессия оборвалась** — откройте блокнот снова, выберите T4 GPU и снова «Выполнить все»:
прогон продолжится с места остановки (готовые примеры лежат на Диске).

Подробная инструкция: `docs/COLAB.md` в репозитории.

In [ ]:
# @title 1. Настройки { display-mode: "form" }
# Модель: 3b — быстро; 7b — основная; 14b — сильнее, но медленнее (для full/max долго).
MODEL = "qwen2.5:7b-instruct"  # @param ["qwen2.5:3b-instruct", "qwen2.5:7b-instruct", "qwen2.5:14b-instruct"] {allow-input: true}
# Вариант: test — 20 атак + 5 чистых (проверка); full — все атаки с автокритерием; max — все примеры.
PRESET = "test"  # @param ["test", "full", "max"]
NUM_CTX = 8192  # @param {type:"integer"}
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
PROGRESS_EVERY = 10  # @param {type:"integer"}

REPO = "https://github.com/bissonette/analysis-of-injections-using-retrieved-context.git"
print(f"Модель: {MODEL} | вариант: {PRESET} | окно контекста: {NUM_CTX} | сохранять на Диск: {SAVE_TO_DRIVE}")

In [ ]:
# @title 2. Проверка видеокарты
import subprocess
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
if r.returncode != 0 or not r.stdout.strip():
    raise RuntimeError("Видеокарта не подключена. Меню «Среда выполнения» → «Сменить среду выполнения» "
                       "→ «T4 GPU» → «Сохранить», затем снова «Выполнить все». "
                       "Если GPU нет в списке или «нет свободных GPU» — попробуйте позже.")
print("Видеокарта:", r.stdout.strip())

In [ ]:
# @title 3. Google Диск — куда сохранять результаты
import os, re
BASE = "/content" if os.path.isdir("/content") else os.getcwd()   # Colab: /content, Kaggle: /kaggle/working
PROJECT = f"{BASE}/project"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_DIR = "/content/drive/MyDrive/vkr_live_runs"
else:
    RESULTS_DIR = f"{BASE}/vkr_live_runs"   # пропадёт при обрыве сессии!
os.makedirs(RESULTS_DIR, exist_ok=True)
TAG = re.sub(r"[^A-Za-z0-9._-]+", "_", MODEL)
OUT = f"{RESULTS_DIR}/endtoend_real_{PRESET}_{TAG}.json"
CHECKPOINT = OUT[:-5] + ".jsonl"
print("Сводка:              ", OUT)
print("Подробно по примерам:", CHECKPOINT)
if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT, encoding="utf-8") as f:
        print(f"Найден прошлый прогон: {sum(1 for _ in f)} примеров готово — продолжим с этого места.")

In [ ]:
# @title 4. Установка Ollama (1–2 мин)
import shutil
if shutil.which("ollama") is None:
    !apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd pciutils lshw > /dev/null 2>&1
    !curl -fsSL https://ollama.com/install.sh | sh
else:
    print("Ollama уже установлена")
if shutil.which("ollama") is None:
    raise RuntimeError("Ollama не установилась — пришлите вывод этой ячейки.")
!ollama --version

In [ ]:
# @title 5. Запуск сервера Ollama
import os, subprocess, time, urllib.request

def ollama_alive():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=3)
        return True
    except Exception:
        return False

if not ollama_alive():
    env = dict(os.environ, OLLAMA_KEEP_ALIVE="-1")   # держать модель в памяти весь прогон
    log = open(f"{BASE}/ollama.log", "a")
    subprocess.Popen(["ollama", "serve"], stdout=log, stderr=log, env=env, start_new_session=True)
    for _ in range(90):
        if ollama_alive():
            break
        time.sleep(1)
if not ollama_alive():
    print(open(f"{BASE}/ollama.log", encoding="utf-8", errors="replace").read()[-3000:])
    raise RuntimeError("Сервер Ollama не запустился — пришлите вывод этой ячейки.")
print("Сервер Ollama работает")

In [ ]:
# @title 6. Скачивание модели (3B ≈ 2 ГБ, 7B ≈ 5 ГБ, 14B ≈ 9 ГБ)
!ollama pull {MODEL} 2>&1 | tail -n 3

In [ ]:
# @title 7. Проверка: модель отвечает и работает на видеокарте
!ollama run {MODEL} "Reply with the single word OK."
!ollama ps
print("\nВ столбце PROCESSOR должно быть «100% GPU».")

In [ ]:
# @title 8. Проект и бенчмарки (1–3 мин)
import os
os.chdir(BASE)
if not os.path.isdir(f"{PROJECT}/.git"):
    !git clone -q --depth 1 {REPO} {PROJECT}
else:
    !git -C {PROJECT} pull -q
os.chdir(PROJECT)
!pip -q install -r requirements.txt

BENCHMARKS = {
    "BIPIA": "https://github.com/microsoft/BIPIA.git",
    "PIArena": "https://github.com/sleeepeer/PIArena.git",
    "Should-It-Be-Executed-Or-Processed": "https://github.com/egozverev/Should-It-Be-Executed-Or-Processed.git",
    "InjecAgent": "https://github.com/uiuc-kang-lab/InjecAgent.git",
    "RaccoonBench": "https://github.com/M0gician/RaccoonBench.git",
    "indirect-pia-detection": "https://github.com/LukeChen-go/indirect-pia-detection.git",
}
os.makedirs("ext", exist_ok=True)
for name, url in BENCHMARKS.items():
    if not os.path.isdir(f"ext/{name}"):
        print("клонирую", name)
        !git clone -q --depth 1 {url} ext/{name}
missing = [n for n in BENCHMARKS if not os.path.isdir(f"ext/{n}")]
if missing:
    raise RuntimeError(f"Не скачались бенчмарки: {missing} — перезапустите эту ячейку.")
print("Бенчмарки на месте:", ", ".join(sorted(BENCHMARKS)))

In [ ]:
# @title 9. Прогон (если прервался — просто запустите снова, продолжит с места)
%cd {PROJECT}
!python scripts/experiment_endtoend.py --ext ext --backend ollama --model {MODEL} --preset {PRESET} --num-ctx {NUM_CTX} --progress-every {PROGRESS_EVERY} --out "{OUT}"

In [ ]:
# @title 10. Итог
import json, os
if not os.path.exists(OUT):
    raise RuntimeError("Сводки ещё нет — прогон не завершился. Запустите ячейку 9 снова.")
with open(OUT, encoding="utf-8") as f:
    r = json.load(f)
print(f"Модель: {r['llm']} | атак: {r['n_attacks']} | чистых: {r['n_clean']} | ошибок: {r['n_errors']}")
print(f"Успешность атак: без фильтра {r['asr_no_filter']:.2f} -> с фильтром {r['asr_filter']:.2f} "
      f"(снижение {r['asr_reduction']*100:.0f}%)")
print(f"Ложные срабатывания на чистых: {r['clean_blocked_rate']:.3f} | ответы сохранены: {r['clean_answer_preserved']:.3f}")
print("\nПо видам влияния:")
for k, m in r["by_permission"].items():
    print(f"  {k:14} {m['asr_no_filter']:.2f} -> {m['asr_filter']:.2f}  (n={m['n']})")
print("\nПрислать эти два файла (они на Google Диске, папка vkr_live_runs):")
print(" ", OUT)
print(" ", CHECKPOINT)

In [ ]:
# @title 11. (Необязательно) Скачать файлы на компьютер
DOWNLOAD = False  # @param {type:"boolean"}
if DOWNLOAD:
    from google.colab import files
    files.download(OUT)
    files.download(CHECKPOINT)
else:
    print("Файлы уже лежат на Google Диске. Чтобы скачать сюда — отметьте DOWNLOAD и запустите ячейку.")